# 🌾 Agro World - ML Model Training

This notebook documents the end-to-end data processing, exploratory data analysis (EDA), model training, evaluation, and serialization for:
1. **Crop Recommendation Model** (Random Forest on 2,200 agricultural samples across 22 crop classes)
2. **Fertilizer Recommendation Model** (Random Forest on soil nutrient & crop type data with cross-validation and overfitting analysis)

---

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. 🌱 Crop Recommendation Model

### 1.1 Dataset Overview
The crop recommendation dataset contains 2,200 rows and 8 columns:
- **Features**: Nitrogen (N), Phosphorus (P), Potassium (K), Temperature (°C), Humidity (%), Soil pH, and Rainfall (mm)
- **Target**: Crop label (22 unique crops, exactly 100 samples per crop)

In [ ]:
crop_df = pd.read_csv('../dataset/Crop_recommendation.csv')
print("Shape:", crop_df.shape)
crop_df.head()

In [ ]:
print("Unique crops:", crop_df['label'].nunique())
print("Missing values:", crop_df.isnull().sum().sum())
crop_df.describe()

### 1.2 Feature Scaling and Splitting
We use `StandardScaler` to normalize the 7 features and split 80% train / 20% test with stratified sampling.

In [ ]:
feature_cols = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
CROP_CLASSES = sorted(crop_df['label'].unique())
crop_to_idx = {crop: i for i, crop in enumerate(CROP_CLASSES)}

X = crop_df[feature_cols]
y = crop_df['label'].map(crop_to_idx)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")

### 1.3 Model Training & Evaluation

In [ ]:
crop_rf = RandomForestClassifier(n_estimators=100, random_state=42)
crop_rf.fit(X_train_scaled, y_train)

train_acc = accuracy_score(y_train, crop_rf.predict(X_train_scaled))
test_acc = accuracy_score(y_test, crop_rf.predict(X_test_scaled))

print(f"Crop Model Train Accuracy: {train_acc * 100:.2f}%")
print(f"Crop Model Test Accuracy:  {test_acc * 100:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, crop_rf.predict(X_test_scaled), target_names=CROP_CLASSES))

---
## 2. 🧪 Fertilizer Recommendation Model

### 2.1 Dataset Overview
The fertilizer dataset contains **99 samples** across 7 fertilizer types:
- Numerical: Temperature, Humidity, Moisture, Nitrogen (N), Potassium (K), Phosphorus (P)
- Categorical: Soil Type, Crop Type

> ⚠️ **Honest Evaluation Note:** With only 99 samples, high accuracy on an 80/20 holdout split is prone to lucky splits and overfitting. We evaluate using **5-Fold Stratified Cross-Validation** to measure generalization variance.

In [ ]:
fert_df = pd.read_csv('../dataset/Fertilizer Prediction.csv')
print("Shape:", fert_df.shape)
fert_df.head()

In [ ]:
# Feature encoding consistent with web application
SOIL_TYPES = ['Black', 'Clayey', 'Loamy', 'Red', 'Sandy']
CROP_TYPES_FERT = ['Barley', 'Cotton', 'Ground Nuts', 'Maize', 'Millets', 'Oil seeds', 'Paddy', 'Pulses', 'Sugarcane', 'Tobacco', 'Wheat']
FERTILIZER_CLASSES = ['10-26-26', '14-35-14', '17-17-17', '20-20', '28-28', 'DAP', 'Urea']

soil_to_idx = {s: i for i, s in enumerate(SOIL_TYPES)}
crop_to_idx = {c: i for i, c in enumerate(CROP_TYPES_FERT)}
fert_to_idx = {f: i for i, f in enumerate(FERTILIZER_CLASSES)}

# Note CSV feature order: Temparature, Humidity , Moisture, Nitrogen, Potassium, Phosphorous (N, K, P)
num_cols = ['Temparature', 'Humidity ', 'Moisture', 'Nitrogen', 'Potassium', 'Phosphorous']
X_num = fert_df[num_cols].values
soil_enc = fert_df['Soil Type'].map(soil_to_idx).values.reshape(-1, 1)
crop_enc = fert_df['Crop Type'].map(crop_to_idx).values.reshape(-1, 1)
y_fert = fert_df['Fertilizer Name'].map(fert_to_idx).values

fert_scaler = StandardScaler()
X_num_scaled = fert_scaler.fit_transform(X_num)
X_fert = np.concatenate([X_num_scaled, soil_enc, crop_enc], axis=1)

In [ ]:
# 5-Fold Stratified Cross Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_clf = RandomForestClassifier(n_estimators=100, random_state=42)
scores = cross_val_score(cv_clf, X_fert, y_fert, cv=skf)

print(f"5-Fold CV Accuracy: {scores.mean() * 100:.2f}% (std: {scores.std() * 100:.2f}%)")
for fold_idx, score in enumerate(scores, 1):
    print(f"  Fold {fold_idx}: {score * 100:.1f}%")

### 2.2 Model Serialization & Export
Both models and scalers are saved to `models/ML_models/` for consumption by `app.py`.

In [ ]:
os.makedirs('../models/ML_models', exist_ok=True)

# Save Crop artifacts
with open('../models/ML_models/crop_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('../models/ML_models/crop_model.pkl', 'wb') as f:
    pickle.dump(crop_rf, f)

# Fit & Save Fertilizer artifacts on full dataset
fert_model = RandomForestClassifier(n_estimators=100, random_state=42)
fert_model.fit(X_fert, y_fert)

with open('../models/ML_models/fertilizer_scaler.pkl', 'wb') as f:
    pickle.dump(fert_scaler, f)
with open('../models/ML_models/fertilizer_model.pkl', 'wb') as f:
    pickle.dump(fert_model, f)

print("All ML models successfully trained and exported!")